In [1]:
# 1 - CREATE - 1_INSTRUCTION_ALL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import glob, os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '2_INPUT/riscv-unified-db/backends/instructions_appendix/all_instructions.golden.adoc'
FOLDER_INPUT = '2_INPUT/riscv-opcodes/extensions'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_BLOCK_SORT_KEY(STRING_BLOCK: str):
    MATCH = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    STRING_NAME = MATCH.group(1).strip() if MATCH else STRING_BLOCK
    return FUNCTION_NATURAL_SORT_KEY(STRING_NAME)

def FUNCTION_NORMALIZE_SUFFIX_FORM(STRING_TOKEN):
    for EACH_PATTERN_SUFFIX, EACH_STRING_KIND in ((r'([a-z_]*imm)(\d+)(?:sp)?(hi|lo)', None), (r'([a-z_]*imm)(\d+)sp(?:_s)?', 'sp'), (r'([a-z_]*imm)(\d+)(hi|lo)', None)):
        MATCH_SUFFIX = re.fullmatch(EACH_PATTERN_SUFFIX, STRING_TOKEN, re.I)
        if MATCH_SUFFIX:
            return (f'{MATCH_SUFFIX.group(1)}{MATCH_SUFFIX.group(2)}', (EACH_STRING_KIND or MATCH_SUFFIX.group(3).lower()))
    if re.fullmatch(r'[a-z_]*imm\d+$', STRING_TOKEN, re.I):
        return STRING_TOKEN, 'plain'
    return None, None

def FUNCTION_CHOOSE_IMM(LIST_OPERAND):
    LIST_IMM = [EACH_TOKEN for EACH_TOKEN in LIST_OPERAND if REGEX_TOKEN_IMM.fullmatch(EACH_TOKEN)]
    if not LIST_IMM:
        return None
    MAP_GROUP = {}
    for EACH_TOKEN in LIST_IMM:
        STRING_BASE, STRING_KIND = FUNCTION_NORMALIZE_SUFFIX_FORM(EACH_TOKEN)
        if STRING_BASE:
            MAP_GROUP.setdefault(STRING_BASE, set()).add(STRING_KIND)
    for EACH_STRING_PREFERENCE in ('hi lo', 'sp', 'plain'):
        for EACH_STRING_BASE, EACH_SET_KIND in MAP_GROUP.items():
            if all(EACH_PREF in EACH_SET_KIND for EACH_PREF in EACH_STRING_PREFERENCE.split()):
                return EACH_STRING_BASE
    STRING_BASE, _ = FUNCTION_NORMALIZE_SUFFIX_FORM(LIST_IMM[0])
    return STRING_BASE or LIST_IMM[0]

def FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN):
    return [EACH_TOKEN for EACH_TOKEN in LIST_TOKEN if not REGEX_TOKEN_ENCODING.match(EACH_TOKEN) and '=' not in EACH_TOKEN and '..' not in EACH_TOKEN and '::' not in EACH_TOKEN]

def FUNCTION_PROMPT(MAP_IMM, STRING_MNEMONIC, LIST_OPERAND):
    STRING_IMM = FUNCTION_CHOOSE_IMM(LIST_OPERAND)
    if STRING_IMM and STRING_MNEMONIC not in MAP_IMM:
        MAP_IMM[STRING_MNEMONIC] = STRING_IMM
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_TOKEN_IMM = re.compile(r'(?:[a-z_]*imm\d+(?:hi|lo|sphi|splo|sp(?:_s)?)?|zimm(?:\d+)?|shamt\d+)$', re.I)
REGEX_TOKEN_ENCODING = re.compile(r'(?:^\d+\.\.\d+=|^\d+=|^0x[0-9a-f]+$|^[01]+$|^\d+\.\.\d+$)', re.I)
MAP_IMM = {}
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_LINE = FILE_HANDLE.readlines()

LIST_BLOCK = [(f"INSTRUCTION: {EACH_LINE_NEXT.split()[0].upper()}\nFORMAT: " f"{EACH_LINE_NEXT.strip().replace('xd','rd').replace('xs1','rs1').replace('xs2','rs2')}") for EACH_LINE_CURRENT, EACH_LINE_NEXT in zip(LIST_LINE, LIST_LINE[1:]) if EACH_LINE_CURRENT.startswith('Assembly::')]

for EACH_FILE_PATH in (EACH_PATH for EACH_PATH in glob.glob(os.path.join(FOLDER_INPUT, 'rv*')) if os.path.isfile(EACH_PATH)):
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        for EACH_RAW_LINE in FILE_HANDLE:
            STRING_LINE = EACH_RAW_LINE.strip()
            if not STRING_LINE:
                continue
            if '->' in STRING_LINE:
                try:
                    _, STRING_RIGHT = STRING_LINE.split('->', 1)
                    LIST_TOKEN = STRING_RIGHT.split()
                    if LIST_TOKEN:
                        FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[0].split(':')[-1], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[1:]))
                except ValueError:
                    pass
                continue
            LIST_TOKEN = STRING_LINE.split()
            if not LIST_TOKEN:
                continue
            if LIST_TOKEN[0] == '$pseudo_op' and len(LIST_TOKEN) >= 3:
                FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[2], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[3:]))
                continue
            FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[0].split(':')[-1], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[1:]))

for EACH_STRING_BLOCK in LIST_BLOCK:
    STRING_INSTRUCTION = EACH_STRING_BLOCK.splitlines()[0].split(':', 1)[1].strip()
    STRING_FORMAT = EACH_STRING_BLOCK.splitlines()[1].split(':', 1)[1].strip()
    STRING_IMM = MAP_IMM.get(STRING_INSTRUCTION.lower())
    if STRING_IMM:
        STRING_FORMAT = re.sub(r'\bimm\b(?=\()', STRING_IMM, STRING_FORMAT, flags=re.I)
        STRING_FORMAT = re.sub(r'\bimm\b(?!\()', STRING_IMM, STRING_FORMAT, flags=re.I)
    for EACH_PATTERN_LABEL in (r'\bbimm\d+(hi|lo)?\b', r'\bjimm\d+\b', r'\bc_bimm\d+(hi|lo)?\b', r'\bc_imm12\b'):
        STRING_FORMAT = re.sub(EACH_PATTERN_LABEL, 'label', STRING_FORMAT, flags=re.I)
    if STRING_INSTRUCTION.upper() in ('AUIPC', 'LUI'):
        STRING_FORMAT = STRING_FORMAT.replace('imm20', 'uimm20')
    LIST_BLOCK_FINAL.append(f'INSTRUCTION: {STRING_INSTRUCTION.upper()}\nFORMAT: {STRING_FORMAT}')

LIST_BLOCK_FINAL = [EACH_BLOCK.replace('ssamoswap.d rd, rs2, rs1', 'ssamoswap.d rd, rs2, (rs1)').replace('ssamoswap.w rd, rs2, rs1', 'ssamoswap.w rd, rs2, (rs1)') for EACH_BLOCK in LIST_BLOCK_FINAL]

LIST_BLOCK_FINAL.sort(key=FUNCTION_BLOCK_SORT_KEY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_BLOCK_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_INSTRUCTION_ALL.txt - END

0.15 second(s)
====
DONE
====


In [2]:
# 2 - CREATE - 2_INSTRUCTION_SOME.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_INSTRUCTION_WANTED = ['FCLASS.D', 'FADD.D', 'FCVT.D.H', 'FMIN.D', 'FSUB.D', 'VAADD.VV', 'VADC.VIM', 'VADD.VI', 'VSUB.VV', 'VAND.VI']
IS_USING_EVERYTHING = False
# VARIABLE - FINISH

# CODE - BEGIN
LIST_INSTRUCTION_WANTED.sort(key=FUNCTION_NATURAL_SORT_KEY)

if IS_USING_EVERYTHING:
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        STRING_CONTENT = FILE_HANDLE.read().strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)
else:
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        LIST_LINE = FILE_HANDLE.readlines()
    LIST_WANTED = sorted({EACH_INSTRUCTION.upper() for EACH_INSTRUCTION in LIST_INSTRUCTION_WANTED}, key=FUNCTION_NATURAL_SORT_KEY)
    LIST_SELECTED = []
    for EACH_STRING_WANTED in LIST_WANTED:
        for EACH_INDEX_LINE, EACH_STRING_LINE in enumerate(LIST_LINE):
            if EACH_STRING_LINE.strip() == f'INSTRUCTION: {EACH_STRING_WANTED}':
                LIST_SELECTED.append(f'{EACH_STRING_LINE.strip()}\n{LIST_LINE[EACH_INDEX_LINE + 1].strip()}')
                break
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write('\n\n'.join(LIST_SELECTED))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_INSTRUCTION_SOME.txt - END

0.00 second(s)
====
DONE
====


In [3]:
# 3 - CREATE - 3_INSTRUCTION_BOOLEAN.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '3_SPECIFICATION/2_SPLIT'
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/3_INSTRUCTION_BOOLEAN.txt'
FILE_PROMPT = '4_INSTRUCTION/2_HISTORY/1_HISTORY_INSTRUCTION_BOOLEAN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
CONTEXT = 16384
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
WIDTH = 120
DIVIDER_LINE = ('#' * WIDTH + '\n') * 3
DIVIDER_LINE = DIVIDER_LINE.rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_OUTPUT_LINE = []
LIST_HISTORY = []
GLOBAL_NUMBER = 1
IS_LOCAL = False
SET_COMPRESS = {'add', 'addi', 'addi16sp', 'addi4spn', 'addiw', 'addw', 'and', 'andi', 'beqz', 'bnez', 'ebreak', 'fld', 'fldsp', 'flw', 'flwsp', 'fsd', 'fsdsp', 'fsw', 'fswsp', 'j', 'jal', 'jalr', 'jr', 'lbu', 'ld', 'ldsp', 'lh', 'lhu', 'li', 'lui', 'lw', 'lwsp', 'mop.n', 'mul', 'mv', 'nop', 'not', 'ntl.all', 'ntl.p1', 'ntl.pall', 'ntl.s1', 'or', 'sb', 'sd', 'sdsp', 'sext.b', 'sext.h', 'sh', 'slli', 'srai', 'srli', 'sub', 'subw', 'sw', 'swsp', 'xor', 'zext.b', 'zext.h', 'zext.w'}
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_INSTRUCTION = [EACH_LINE.split(': ')[1].strip() for EACH_LINE in FILE_HANDLE if EACH_LINE.startswith('INSTRUCTION: ')]

for EACH_INSTRUCTION in LIST_INSTRUCTION:
    SET_KEY = {EACH_INSTRUCTION, EACH_INSTRUCTION.lower()}
    if EACH_INSTRUCTION.startswith('AMO'):
        SET_KEY.add('AMO')
    elif '.' in EACH_INSTRUCTION and EACH_INSTRUCTION.index('.') >= 3:
        SET_KEY.add(EACH_INSTRUCTION.split('.', 1)[0])
    LIST_PATH = []
    for EACH_FILE_NAME in sorted(os.listdir(FOLDER_INPUT), key=FUNCTION_NATURAL_SORT_KEY):
        if not EACH_FILE_NAME.endswith('.txt'):
            continue
        FILE_PATH = os.path.join(FOLDER_INPUT, EACH_FILE_NAME)
        with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            if any(any(EACH_KEY in EACH_LINE for EACH_KEY in SET_KEY) for EACH_LINE in FILE_HANDLE):
                LIST_PATH.append(FILE_PATH)
    LIST_PATH.sort(key=lambda EACH_PATH_VALUE: int(EACH_PATH_VALUE.split('SPECIFICATION_')[1].split('.txt')[0]))
    LOCAL_NUMBER = 1
    for EACH_FILE_PATH in LIST_PATH:
        with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
            STRING_SPEC = FILE_HANDLE.read()
        STRING_EXCEPTION = (f'Remember, {EACH_INSTRUCTION}, not C.{EACH_INSTRUCTION}! ' f'If the text contains any information about C.{EACH_INSTRUCTION}, output NO! ' if EACH_INSTRUCTION.lower() in SET_COMPRESS else '')
        STRING_PROMPT = (f'For RISC-V, does this text below contain any information about ' f'{EACH_INSTRUCTION} instruction? {STRING_EXCEPTION}' f'If the text contains any information about HINT instructions, ' f'output NO! Output YES or NO only.\n\nThe text:\n\n{STRING_SPEC}')
        if IS_LOCAL:
            STRING_RESPONSE = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        else:
            STRING_RESPONSE = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
        STRING_RESPONSE = FUNCTION_CLEAN_TEXT(STRING_RESPONSE)
        STRING_INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {EACH_INSTRUCTION}
{'=' * (WIDTH // 2)}
=======
PROMPT:
=======
{'=' * (WIDTH // 2)}
'''
        STRING_ENDING = f'''
{'=' * (WIDTH // 2)}
=========
RESPONSE:
=========
{'=' * (WIDTH // 2)}
{STRING_RESPONSE}
'''
        LIST_HISTORY.append(f"{f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')}\n" f"{STRING_INTRO}{STRING_PROMPT}{STRING_ENDING}" f"{f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')}{SUFFIX}")
        if 'YES' in STRING_RESPONSE:
            LIST_OUTPUT_LINE.append(f'{EACH_INSTRUCTION} - {EACH_FILE_PATH}')
        GLOBAL_NUMBER += 1
        LOCAL_NUMBER += 1

STRING_PROMPT_HISTORY = ''.join(LIST_HISTORY).removesuffix(SUFFIX)

with open(FILE_PROMPT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n'.join(LIST_OUTPUT_LINE))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_INSTRUCTION_BOOLEAN.txt - END

19 minute(s) 26.55 second(s)
====
DONE
====


In [4]:
# 4 - CREATE - 4_INSTRUCTION_RULE.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/3_INSTRUCTION_BOOLEAN.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/4_INSTRUCTION_RULE.txt'
FILE_PROMPT_1 = '4_INSTRUCTION/2_HISTORY/2_HISTORY_INSTRUCTION_RULE_1.txt'
FILE_PROMPT_2 = '4_INSTRUCTION/2_HISTORY/3_HISTORY_INSTRUCTION_RULE_2.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))

def FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    if IS_LOCAL:
        return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()
    return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}).message.content.strip()

def FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE, STRING_SUFFIX):
    HALF_WIDTH = WIDTH // 2
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + STRING_SUFFIX
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODEL = 'gpt-oss:120b'
TEMPERATURE = 1
API_KEY = '1dd95d27ffbc4b93a048c6ab1fbaa2e2.asxCCp5nO-KXM8jtg-TKNtw4'
CLIENT = Client(host='https://ollama.com', headers={'Authorization': API_KEY})
CONTEXT = 65536
IS_LOCAL = False
WIDTH = 120
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_OUTPUT_CONTENT = []
MAP_INSTRUCTION_RULE = defaultdict(list)
LIST_OUTPUT_FINAL = []
LIST_OUTPUT_SUPER_FINAL = []
STRING_PROMPT_HISTORY_1 = ''
STRING_PROMPT_HISTORY_2 = ''
GLOBAL_NUMBER_1 = 1
LOCAL_NUMBER_1 = 1
GLOBAL_NUMBER_2 = 1
LOCAL_NUMBER_2 = 1
INSTRUCTION_INITIAL = None
MAP_REGISTER_TO_LIST_SPEC = defaultdict(list)
LIST_REGISTER_ORDER = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_PAIR = [EACH_LN.strip().split(' - ', 1) for EACH_LN in FILE_HANDLE if EACH_LN.strip()]

for EACH_INSTRUCTION, EACH_FILE_PATH in LIST_PAIR:
    if EACH_INSTRUCTION not in MAP_REGISTER_TO_LIST_SPEC:
        LIST_REGISTER_ORDER.append(EACH_INSTRUCTION)
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        MAP_REGISTER_TO_LIST_SPEC[EACH_INSTRUCTION].append(FILE_HANDLE.read())

for EACH_INSTRUCTION in LIST_REGISTER_ORDER:
    TOTAL_STRING_SPEC = '\n\n###\n\n'.join(MAP_REGISTER_TO_LIST_SPEC[EACH_INSTRUCTION]).strip()
    LOCAL_COUNTER_1 = 1
    LOCAL_COUNTER_2 = 1
    INSTRUCTION_NAME = EACH_INSTRUCTION
    
    # PROMPT 1 - BEGIN
    STRING_PROMPT_1 = f'''For RISC-V, please extract all possible information about {INSTRUCTION_NAME} instruction from the text below. The information must be only about {INSTRUCTION_NAME} instruction, not other things. Extract ONLY explicit, testable, normative description about architectural behavior.

Please output in paragraphs. If you want, you can include bullet or numbered points in your output. You can also include tables. However, do not give me any introduction in your output. You just need to output the information that you extract.

If your output includes some sentences that refer to some tables or figures, include those tables or figures too.

Remember, output must be ASCII-only text.

If it happens to be there is no information about {INSTRUCTION_NAME} instruction from the text below, output NONE

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Your description must be very clear and unambiguous. If you want, you can output in bullet points also.

Use the exact terminology and wording used in the text below; do not substitute synonyms or paraphrase architectural terms.

The text below:

{TOTAL_STRING_SPEC}'''
    STRING_RESPONSE_1 = FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_1, TEMPERATURE, CONTEXT)
    if STRING_RESPONSE_1.strip() == 'NONE':
        continue
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(STRING_RESPONSE_1)
    STRING_PROMPT_HISTORY_1 += FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER_1, LOCAL_NUMBER_1, INSTRUCTION_NAME, STRING_PROMPT_1, STRING_RESPONSE_1, SUFFIX)
    # PROMPT 1 - FINISH

    # PROMPT 2 - BEGIN
    STRING_PROMPT_2 = f'''For RISC-V, for {INSTRUCTION_NAME} instruction, rewrite the following description as a list of rules. Each rule must explain about {INSTRUCTION_NAME} instruction. Essentially, for each rule, it is about {INSTRUCTION_NAME} instruction, not other things.

Do not lose any information about {INSTRUCTION_NAME} instruction from the description. Read the text or the description carefully. Understand the context and the sentences. Do not lose any important information.

For each rule, do not use any internal references like "According to Table 7," or "see Figure 3." Each rule must be one sentence only. Each rule must stand fully on its own without cross-references. Do not quote or refer to other rules. Essentially, each rule is one idea. Avoid a rule like this: "The instruction being described is {INSTRUCTION_NAME}."

Each rule should become a single testable statement. Each rule should be a separate atomic rule, each expressing only one fact. Each rule needs to be clear and detailed. Extract ONLY explicit, testable, normative rules about architectural behavior. A rule MUST describe behavioral semantics, not naming, encoding, or syntax.

Remember, rules must describe architectural cause -> effect semantics (event -> architectural outcome).

Remember, if a sentence contains multiple distinct behaviors or enumerated cases, split them into multiple rules so each rule expresses only one case.

Exclude purely structural or descriptive properties (such as register width, naming, or existence) unless they imply observable architectural behavior.

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

OUTPUT FORMAT:
+ Each rule must begin with the word "RULE", followed by its number (e.g., "RULE 1: ", "RULE 2: ", "RULE 3: ", ...).
+ Do not give me any introduction.
+ Each rule must contain the word {INSTRUCTION_NAME}.

Remember, output must be ASCII-only text.

The following description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_2, TEMPERATURE, CONTEXT))
    STRING_PROMPT_HISTORY_2 += FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER_2, LOCAL_NUMBER_2, INSTRUCTION_NAME, STRING_PROMPT_2, STRING_RESPONSE_2, SUFFIX)
    # PROMPT 2 - FINISH

    # FINALIZING - BEGIN
    STRING_RESPONSE_FINISH = STRING_RESPONSE_2
    LIST_LINE_RULE = [EACH_LN.strip() for EACH_LN in STRING_RESPONSE_FINISH.splitlines() if EACH_LN.strip()]
    LIST_RULE = []
    for EACH_LINE_RULE in LIST_LINE_RULE:
        if re.match(r'^RULE\s+\d+:', EACH_LINE_RULE, re.IGNORECASE):
            EACH_LINE_RULE = re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LINE_RULE)
            LIST_RULE.append(EACH_LINE_RULE)
    LIST_LINE_FIXED = []
    for EACH_LINE_RULE in LIST_RULE:
        MATCH_RULE = re.match(r'^\s*RULE\s+(\d+):\s*(.*)', EACH_LINE_RULE)
        if not MATCH_RULE:
            LIST_LINE_FIXED.append(EACH_LINE_RULE)
            continue
        NUMBER_RULE = MATCH_RULE.group(1)
        STRING_REST = MATCH_RULE.group(2)
        STRING_REST = re.sub(rf'^RULE\s+{NUMBER_RULE}:\s*', '', STRING_REST)
        LIST_LINE_FIXED.append(f'RULE {NUMBER_RULE}: {STRING_REST}')
    STRING_CLEANED = '\n'.join(LIST_LINE_FIXED)
    LIST_OUTPUT_CONTENT.append(f'INSTRUCTION: {INSTRUCTION_NAME}\n{STRING_CLEANED}')
    # FINALIZING - FINISH

    # CLEANING UP PROMPT - BEGIN
    GLOBAL_NUMBER_1 += 1
    LOCAL_NUMBER_1 += 1
    GLOBAL_NUMBER_2 += 1
    LOCAL_NUMBER_2 += 1

STRING_PROMPT_HISTORY_1 = STRING_PROMPT_HISTORY_1.removesuffix(SUFFIX)
STRING_PROMPT_HISTORY_2 = STRING_PROMPT_HISTORY_2.removesuffix(SUFFIX)

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_1)

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_2)
# CLEANING UP PROMPT - FINISH

for EACH_STRING_CONTENT in LIST_OUTPUT_CONTENT:
    LIST_LINE = EACH_STRING_CONTENT.strip().splitlines()
    STRING_NAME = LIST_LINE[0].split(':', 1)[1].strip()
    MAP_INSTRUCTION_RULE[STRING_NAME].extend(LIST_LINE[1:])

for EACH_STRING_NAME, EACH_LIST_RULE in MAP_INSTRUCTION_RULE.items():
    LIST_LINE_NEW = [f'INSTRUCTION: {EACH_STRING_NAME}']
    for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(EACH_LIST_RULE, 1):
        STRING_RULE_TEXT = EACH_RULE_LINE.split(':', 1)[1].strip()
        LIST_LINE_NEW.append(f'RULE {EACH_INDEX_RULE}: {STRING_RULE_TEXT}')
    LIST_OUTPUT_FINAL.append('\n'.join(LIST_LINE_NEW))

for EACH_STRING_EACH in LIST_OUTPUT_FINAL:
    LIST_LINE = EACH_STRING_EACH.splitlines()
    STRING_INSTRUCTION_LINE = LIST_LINE[0].strip()
    LIST_RULE = [EACH_LN.strip() for EACH_LN in LIST_LINE[1:] if EACH_LN.strip() and EACH_LN.strip() != 'RULE:']
    LIST_RULE = [re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LN) for EACH_LN in LIST_RULE]
    LIST_RENUMBERED = []
    for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(LIST_RULE, 1):
        LIST_RENUMBERED.append(re.sub(r'RULE\s+\d+:', f'RULE {EACH_INDEX_RULE}:', EACH_RULE_LINE))
    LIST_OUTPUT_SUPER_FINAL.append(f'{STRING_INSTRUCTION_LINE}\n' + '\n'.join(LIST_RENUMBERED))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_OUTPUT_SUPER_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 4_INSTRUCTION_RULE.txt - END

17.92 second(s)
====
DONE
====


In [5]:
# 5 - CREATE - 5_INSTRUCTION_FINAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# PATH - BEGIN
FILE_INPUT_NAME = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
FILE_INPUT_RULE = '4_INSTRUCTION/1_OUTPUT/4_INSTRUCTION_RULE.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/5_INSTRUCTION_FINAL.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_INPUT_NAME, encoding='utf-8') as FILE_HANDLE:
    LIST_NAME = FILE_HANDLE.readlines()

with open(FILE_INPUT_RULE, encoding='utf-8') as FILE_HANDLE:
    LIST_OUTPUT_LINE = FILE_HANDLE.readlines()

for EACH_INDEX_LINE, EACH_STRING_LINE in enumerate(LIST_NAME):
    if EACH_STRING_LINE.startswith('INSTRUCTION: '):
        try:
            INDEX_MATCH = LIST_OUTPUT_LINE.index(EACH_STRING_LINE)
        except ValueError:
            continue
        LIST_OUTPUT_LINE.insert(INDEX_MATCH + 1, LIST_NAME[EACH_INDEX_LINE + 1].rstrip() + '\n')

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(''.join(LIST_OUTPUT_LINE))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 5 - CREATE - 5_INSTRUCTION_FINAL.txt - END

0.00 second(s)
====
DONE
====


In [6]:
# 6 - CREATE - 6_INSTRUCTION_TOTAL.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/5_INSTRUCTION_FINAL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/6_INSTRUCTION_TOTAL.txt'
# PATH - FINISH

# VARIABLE - BEGIN
IS_NEED_INIT = (not os.path.exists(FILE_OUTPUT)) or os.path.getsize(FILE_OUTPUT) == 0
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_INPUT = FILE_HANDLE.read()

if IS_NEED_INIT:
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_INPUT)
else:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_OUTPUT = FILE_HANDLE.read()
    TEXT_COMBINED = TEXT_OUTPUT.strip() + '\n\n' + TEXT_INPUT.strip()
    REGEX_INSTRUCTION = re.compile(r'(?ms)^\s*INSTRUCTION:\s*([^\n]+)\s*\n.*?(?=^\s*INSTRUCTION:\s*[^\n]+\s*\n|\Z)')
    MAP_BY_NAME = {EACH_MATCH.group(1).strip(): EACH_MATCH.group(0) for EACH_MATCH in REGEX_INSTRUCTION.finditer(TEXT_COMBINED)}
    TEXT_FINAL = '\n\n'.join(MAP_BY_NAME[EACH_KEY].strip() for EACH_KEY in sorted(MAP_BY_NAME, key=str.upper)).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_FINAL)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 6 - CREATE - 6_INSTRUCTION_TOTAL.txt - END

0.01 second(s)
====
DONE
====


In [7]:
# 7 - CREATE - 7_INSTRUCTION_SUMMARY.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from typing import Dict, List, Optional, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/7_INSTRUCTION_SUMMARY.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('')
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 7 - CREATE - 7_INSTRUCTION_SUMMARY.txt - END

0.00 second(s)
====
DONE
====
